# 🏥 Doctor's Handwritten Prescription Recognition (DHPRS)
## TrOCR Fine-Tuning on Google Colab (Free T4 GPU)

This notebook trains Microsoft's **TrOCR** (`microsoft/trocr-base-stage1`) on the **RxHandBD** handwritten medical prescription dataset from Zenodo (Record: 18478741).

### Pipeline Overview:
1. **Environment & GPU Setup** (Check NVIDIA T4 GPU)
2. **Google Drive Mount** (For persistent checkpoint storage)
3. **Download RxHandBD Dataset** from Zenodo
4. **Custom PyTorch Dataset** with Aspect-Preserving Padding
5. **Hugging Face Seq2SeqTrainer** with CER (Character Error Rate) Evaluation
6. **Save Best Model Checkpoint** to Google Drive

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Install Required Libraries
!pip install -q transformers==4.40.0 accelerate==0.28.0 evaluate==0.4.1 jiwer==3.0.3 torchvision rapidfuzz albumentations

In [ ]:
# Step 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/DHPRS_Models/trocr-rxhandbd-best'
!mkdir -p "$SAVE_DIR"

In [ ]:
# Step 4: Download RxHandBD from Zenodo
import requests, zipfile, os

DATA_DIR = '/content/rxhandbd'
os.makedirs(DATA_DIR, exist_ok=True)

# Zenodo Record 18478741 direct download
zenodo_api = 'https://zenodo.org/api/records/18478741'
res = requests.get(zenodo_api).json()
print('Files available in Zenodo record:')
for f in res.get('files', []):
    fname = f['key']
    furl = f['links']['self']
    print(f'Downloading {fname}...')
    r = requests.get(furl, stream=True)
    out_path = os.path.join(DATA_DIR, fname)
    with open(out_path, 'wb') as out_f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            out_f.write(chunk)
    if fname.endswith('.zip'):
        with zipfile.ZipFile(out_path, 'r') as z:
            z.extractall(DATA_DIR)
print('Dataset ready in:', DATA_DIR)

In [ ]:
# Step 5: Dataset Loader & Aspect-Preserving Padding
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_NAME = 'microsoft/trocr-base-stage1'
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Special Tokens Configuration
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

def pad_to_aspect_ratio(image, target_size=(384, 384)):
    target_w, target_h = target_size
    img_w, img_h = image.size
    scale = min(target_w / img_w, target_h / img_h)
    new_w = max(1, int(img_w * scale))
    new_h = max(1, int(img_h * scale))
    resized = image.resize((new_w, new_h), Image.Resampling.BILINEAR)
    padded = Image.new('RGB', target_size, (255, 255, 255))
    padded.paste(resized, ((target_w - new_w) // 2, (target_h - new_h) // 2))
    return padded

class PrescriptionWordDataset(Dataset):
    def __init__(self, images_dir, labels_csv, processor, max_target_length=64):
        self.images_dir = images_dir
        self.labels_df = pd.read_csv(labels_csv)
        self.processor = processor
        self.max_target_length = max_target_length
        self.image_col = self.labels_df.columns[0]
        self.label_col = self.labels_df.columns[1]
        self.labels_df = self.labels_df.dropna().reset_index(drop=True)

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        row = self.labels_df.iloc[idx]
        img_name = str(row[self.image_col]).strip()
        label_text = str(row[self.label_col]).strip()
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        padded = pad_to_aspect_ratio(image)
        pixel_values = self.processor(padded, return_tensors='pt').pixel_values.squeeze(0)
        labels = self.processor.tokenizer(label_text, padding='max_length', max_length=self.max_target_length, truncation=True, return_tensors='pt').input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}

In [ ]:
# Step 6: Define Evaluation Metrics (CER & WER)
import evaluate

cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': cer}

In [ ]:
# Step 7: Seq2SeqTrainer Configuration & Training Run
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

# Locate train and test folders
train_img_dir = '/content/rxhandbd/train'
train_csv = '/content/rxhandbd/train_labels.csv'
test_img_dir = '/content/rxhandbd/test'
test_csv = '/content/rxhandbd/test_labels.csv'

train_ds = PrescriptionWordDataset(train_img_dir, train_csv, processor)
eval_ds = PrescriptionWordDataset(test_img_dir, test_csv, processor)

training_args = Seq2SeqTrainingArguments(
    output_dir='./checkpoints',
    predict_with_generate=True,
    evaluation_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=15,
    learning_rate=4e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=processor.feature_extractor,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Step 8: Save Checkpoint to Google Drive
trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f'Model and processor successfully saved to Google Drive at: {SAVE_DIR}')
print('Download this folder and place it in D:\\DHPRS\\models\\checkpoints\\trocr-rxhandbd-best on your local machine!')